# Example Protobuf - Python Benchmarking

This notebook demonstrates generating large datasets of Protocol Buffer messages for benchmarking serialization, deserialization, and conversion to/from Polars DataFrames.

In [1]:
import polars as pl

from example_protobuf.example_protobuf import Person
from example_protobuf.pybindings.person_pb2 import (
    Address as PyAddress,
    Person as PyPerson,
    Status as PyStatus,
)

## Serialized dataset creation using protobuf python bindings

In [2]:
def generate_py_persons(count: int) -> list[PyPerson]:
    """Generate a list of PyPerson objects for benchmarking.

    Args:
        count: Number of PyPerson objects to create

    Returns:
        List of PyPerson objects with realistic data
    """

    persons = []
    for i in range(count):
        address = PyAddress(
            street=f"{100 + (i % 900)} Main Street",
            city=["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"][i % 5],
            zip_code=10000 + (i % 90000),
        )

        previous_addresses = []
        for j in range(i % 3):  # 0-2 previous addresses
            prev_addr = PyAddress(
                street=f"{200 + (j * 100)} Oak Avenue",
                city=["Boston", "Seattle", "Denver"][j % 3],
                zip_code=20000 + (j % 80000),
            )
            previous_addresses.append(prev_addr)

        person = PyPerson(
            name=f"Person_{i}",
            age=18 + (i % 62),  # Ages 18-79
            email=f"person_{i}@example.com" if i % 3 != 0 else "",  # Some without email
            is_active=(i % 2 == 0),
            address=address,
            tags=[f"tag_{i % 10}", f"category_{i % 5}"],
            status=[PyStatus.ACTIVE, PyStatus.INACTIVE, PyStatus.UNKNOWN][i % 3],
            previous_addresses=previous_addresses,
        )
        persons.append(person)

    return persons

In [3]:
# Example: Generate 1 million PyPerson objects
import time

print("Generating 100,000 PyPerson objects...")
start_time = time.time()
persons = generate_py_persons(100_000)
elapsed = time.time() - start_time

print(f"✓ Generated {len(persons):,} persons in {elapsed:.2f} seconds")
print(f"  Rate: {len(persons) / elapsed:,.0f} persons/second")

Generating 100,000 PyPerson objects...
✓ Generated 100,000 persons in 0.44 seconds
  Rate: 226,649 persons/second


In [4]:
# Benchmark serialization of the generated persons
print("Benchmarking serialization...")
start_time = time.time()
serialized_data = pl.DataFrame(
    {"people_encoded": [person.SerializeToString() for person in persons]}
)
elapsed = time.time() - start_time

total_bytes = serialized_data.estimated_size()
print(f"✓ Serialized {len(serialized_data):,} persons in {elapsed:.2f} seconds")
print(f"  Rate: {len(serialized_data) / elapsed:,.0f} persons/second")
print(f"  Total size: {total_bytes:,} bytes ({total_bytes / 1024 / 1024:.2f} MB)")
print(f"  Average size per person: {total_bytes / len(serialized_data):.1f} bytes")

Benchmarking serialization...
✓ Serialized 100,000 persons in 0.05 seconds
  Rate: 2,069,789 persons/second
  Total size: 11,835,339 bytes (11.29 MB)
  Average size per person: 118.4 bytes


## Deserialization

In [5]:
%timeit -n 7 -r 1 [PyPerson.FromString(bytes) for bytes in serialized_data["people_encoded"]]

87.9 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 7 loops each)


In [6]:
%timeit -n 7 -r 1 serialized_data.select(Person.decode(pl.col("people_encoded")))

62.6 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 7 loops each)


In [7]:
%timeit -n 7 -r 1 serialized_data.lazy().select(Person.decode(pl.col("people_encoded"))).collect()

71.1 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 7 loops each)


In [8]:
%timeit -n 7 -r 1 serialized_data.lazy().select(Person.decode(pl.col("people_encoded"))).collect(engine="streaming")

57.5 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 7 loops each)


In [9]:
deserialize_python = [
    PyPerson.FromString(bytes) for bytes in serialized_data["people_encoded"]
]
deserialize_rust = [
    PyPerson(**kwargs)
    for kwargs in serialized_data.select(Person.decode(pl.col("people_encoded")))[
        "people_encoded"
    ]
]
assert all(p == r for p, r in zip(deserialize_python, deserialize_rust))